# Reading and plotting signals
Read recorded signals from ADI Study Watch and BioNomadix devices. 
Plot ECG, PPG, and accelerometer data from the smartwatch, and ECG from BioNomadix.
Signals must be plotted with x-axis in time, in seconds.

In [ ]:
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import scipy
from datetime import timedelta
import pywt

# Import Pandas only for reading Watch CSV File and for plotting description in Q1
import pandas as pd


In [ ]:
def load_bp_csv(filename, start_datetime):
    test = pd.read_csv(filename, skiprows=8, header=1, delimiter='\t')
    test = test.drop(0)
    test = test.drop(test.columns[3], axis=1)  # Drop the first column
    test = test.rename(columns={'milliSec': 'time_ms', 'CH13': 'ecg', 'CH1': 'ppg'})
    test['ecg'] = pd.to_numeric(test['ecg'], errors='coerce')
    test['timestamp'] = pd.to_datetime(start_datetime) + pd.to_timedelta(test['time_ms'], unit='ms')
    return test


def load_watch_ecg(filename):
    df = pd.read_csv(filename, skiprows=2)
    df = df[['Timestamp', 'ECG data']].dropna()
    df.rename(columns={'Timestamp': 'timestamp', 'ECG data': 'ecg'}, inplace=True)
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['ecg'] = pd.to_numeric(df['ecg'], errors='coerce')
    df = df.dropna()

    # Convert to seconds
    df['time_ms'] = (df['timestamp'] - df['timestamp'].iloc[0])
    return df


def load_watch_data(ecg_file, ppg_file):
    ecg_df = pd.read_csv(ecg_file, skiprows=2)
    ppg_df = pd.read_csv(ppg_file, skiprows=2)

    # Rename columns for clarity
    ecg_df.rename(columns={'Timestamp': 'timestamp', 'ECG data': 'ecg'}, inplace=True)
    ppg_df.rename(columns={'PPG Timestamp': 'timestamp', 'PPG data': 'ppg'}, inplace=True)

    # Add column with time in milli seconds
    ecg_df['time_ms'] = (ecg_df['timestamp'] - ecg_df['timestamp'].iloc[0])
    ppg_df['time_ms'] = (ppg_df['timestamp'] - ppg_df['timestamp'].iloc[0])

    latest_start_time = max(ecg_df['timestamp'].iloc[0], ppg_df['timestamp'].iloc[0])
    earliest_end_time = min(ecg_df['timestamp'].iloc[-1], ppg_df['timestamp'].iloc[-1])

    # Filter data to the common time range
    ecg_df = ecg_df[(ecg_df['timestamp'] >= latest_start_time) & (ecg_df['timestamp'] <= earliest_end_time)]
    ppg_df = ppg_df[(ppg_df['timestamp'] >= latest_start_time) & (ppg_df['timestamp'] <= earliest_end_time)]

    # Drop all columns except timestamp, time_ms and ecg/ppg
    ppg_df = ppg_df.drop(columns=['ADXL Timestamp', 'X', 'Y', 'Z'])
    ecg_df = ecg_df.drop(columns=['Seq No.'])
    # Merge ECG and PPG based on the millisecond given
    merged_df = pd.merge_asof(ecg_df.sort_values('timestamp'), ppg_df.sort_values('timestamp'), on='timestamp',
                              direction='nearest')

    merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], unit='ms')
    return merged_df


def plot_watch_bp_ecg(watch_df, bp_data, start_time, end_time, dataset_name='Recording'):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    watch_df.sort_values('time_ms_x', inplace=True)
    bp_data.sort_values('time_ms', inplace=True)

    # Select time frame to plot
    # Discard time frames that are not represented in both recordings
    latest_start_time = max(watch_df['timestamp'].iloc[0], bp_data['timestamp'].iloc[0]) # + timedelta(minutes=5)
    earliest_end_time = min(watch_df['timestamp'].iloc[-1], bp_data['timestamp'].iloc[-1])
    watch_df = watch_df[(watch_df['timestamp'] >= latest_start_time) & (watch_df['timestamp'] <= earliest_end_time)]
    watch_df = watch_df[(watch_df['timestamp'] >= start_time) & (watch_df['timestamp'] <= end_time)]
    bp_data = bp_data[(bp_data['timestamp'] >= latest_start_time) & (bp_data['timestamp'] <= earliest_end_time)]
    bp_data = bp_data[(bp_data['timestamp'] >= start_time) & (bp_data['timestamp'] <= end_time)]

    # Plot ECG from Watch
    ax1.plot(watch_df['timestamp'], watch_df['ecg'], label='ECG from Watch', color='blue')
    ax1.set_ylabel('ECG Amplitude')
    ax1.set_xlabel('Time')
    ax1.set_title(f'{dataset_name} - ECG Signal from Smartwatch')
    ax1.legend()

    # Plot ECG from BioPac
    ax2.plot(bp_data['timestamp'], bp_data['ecg'], label='ECG from BioPac', color='orange')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('ECG Amplitude')
    ax2.set_title(f'{dataset_name} - ECG Signal from BioPac')
    ax2.legend()
    plt.tight_layout()
    plt.show()

def crop_window(data, start_time, end_time):
    data = data[(data['timestamp'] >= start_time) & (data['timestamp'] <= end_time)]
    return data


# Time domain
Consider an ECG signal, using fixed windows of 5s with 50% overlap, compute the following features for a segment of 30s long:
- Mean
- Median
- Variance
- Std Dev
- Skewness
- RMS
- Zero-Crossing Rate
- Energy
- Power

Apply this porcedure to the ECG signals acquired from ADI Study Watch and BioNomadix.

In [ ]:
bp_clean = load_bp_csv('data/assignment_2/recording_1_ecgppg_biopac_clean_data.txt',
                        start_datetime=datetime(2025, 5, 22, 9, 43, 51, 760000) - timedelta(seconds=0.75))
bp_noisy = load_bp_csv('data/assignment_2/recording_2_ecgppg_biopac_noisy.txt',
                        start_datetime=datetime(2025, 5, 22, 9, 54, 40, 285000) - timedelta(seconds=0.4))
watch_clean = load_watch_data('data/assignment_2/recording_1_ecg_watch_clean_data.csv',
                              'data/assignment_2/recording_1_ppg_watch_clean_data.csv')
watch_noisy = load_watch_data('data/assignment_2/recording_2_ecg_watch_noisy.csv',
                              'data/assignment_2/recording_2_ppg_watch_noisy.csv')
watch_motion = load_watch_data('data/assignment_2/recording_3_ecg_watch_movement.csv',
                              'data/assignment_2/recording_3_ppg_watch_movement.csv')
bp_motion = load_bp_csv('data/assignment_2/recording_3_ecgppg_biopac_movement.txt',
                        start_datetime=datetime(2025, 5, 22, 10, 3, 22, 265000) - timedelta(seconds=0.5))

plot_watch_bp_ecg(watch_clean, bp_clean, '2025-05-22 09:45:00', '2025-05-22 09:46:00', dataset_name='Clean')
plot_watch_bp_ecg(watch_noisy, bp_noisy, '2025-05-22 09:55:10', '2025-05-22 09:56:10', dataset_name='Noisy')
plot_watch_bp_ecg(watch_motion, bp_motion, '2025-05-22 10:04:00', '2025-05-22 10:05:30', dataset_name='Motion')


In [ ]:
#calculate measurements from watch and bp data

def plot_data_descriptions(watch_df, bp_df, dataset_name='Give Name Please'):
    description_df.loc[len(description_df)] = [
        dataset_name + ' Watch',
        watch_df['ecg'].mean(),
        watch_df['ecg'].median(),
        watch_df['ecg'].var(),
        watch_df['ecg'].std(),
        watch_df['ecg'].skew(),
        np.sqrt(np.mean(watch_df['ecg'] ** 2)),
        np.mean(np.diff(np.sign(watch_df['ecg'])) != 0),
        np.sum(watch_df['ecg'] ** 2),
        np.mean(watch_df['ecg'] ** 2)
    ]
    description_df.loc[len(description_df)] = [
        dataset_name + ' BP',
        bp_df['ecg'].mean(),
        bp_df['ecg'].median(),
        bp_df['ecg'].var(),
        bp_df['ecg'].std(),
        bp_df['ecg'].skew(),
        np.sqrt(np.mean(bp_df['ecg'] ** 2)),
        np.mean(np.diff(np.sign(bp_df['ecg'])) != 0),
        np.sum(bp_df['ecg'] ** 2),
        np.mean(bp_df['ecg'] ** 2)
    ]

In [ ]:
# Data Description DF
description_df = pd.DataFrame({
    'Dataset Name': [], 'Mean': [], 'Median': [], 'Variance': [], 'Std Dev': [], 'Skewness': [], 'RMS': [], 'Zero-Crossing Rate': [],
    'Energy': [], 'Power': []
})

# crop 1 minute window for q1a
minute_watch_clean = crop_window(watch_clean,'2025-05-22 09:45:00', '2025-05-22 09:46:00')
minute_watch_noisy =crop_window(watch_noisy, '2025-05-22 09:55:10', '2025-05-22 09:56:10')
minute_bp_clean = crop_window(bp_clean,'2025-05-22 09:45:00', '2025-05-22 09:46:00')
minute_bp_noisy =crop_window(bp_noisy, '2025-05-22 09:55:10', '2025-05-22 09:56:10')

plot_data_descriptions(minute_watch_noisy, minute_bp_noisy, dataset_name='Noisy')
plot_data_descriptions(minute_watch_clean, minute_bp_clean, dataset_name='Clean')

# print results for q1a
description_df


In [ ]:
# start q1b
# generate timestamps for cropping 10s windows
clean_timestamps = ['2025-05-22 09:45:00', '2025-05-22 09:45:10', '2025-05-22 09:45:20', '2025-05-22 09:45:30', '2025-05-22 09:45:40', '2025-05-22 09:45:50', '2025-05-22 09:46:00']

overlapping_clean_timestamps = ['2025-05-22 09:45:00', '2025-05-22 09:45:10', '2025-05-22 09:45:07.5', '2025-05-22 09:45:17.5', '2025-05-22 09:45:15', '2025-05-22 09:45:25', '2025-05-22 09:45:22.5', '2025-05-22 09:45:32.5', '2025-05-22 09:45:30', '2025-05-22 09:45:40', '2025-05-22 09:45:37.5', '2025-05-22 09:45:47.5', '2025-05-22 09:46:45','2025-05-22 09:46:55', '2025-05-22 09:46:52.5', '2025-05-22 09:47:02.5']

noisy_timestamps = ['2025-05-22 09:55:00', '2025-05-22 09:55:10', '2025-05-22 09:55:20', '2025-05-22 09:55:30', '2025-05-22 09:55:40', '2025-05-22 09:55:50', '2025-05-22 09:56:00']

overlapping_noisy_timestamps = ['2025-05-22 09:55:00', '2025-05-22 09:55:10', '2025-05-22 09:55:07.5', '2025-05-22 09:55:17.5', '2025-05-22 09:55:15', '2025-05-22 09:55:25', '2025-05-22 09:55:22.5', '2025-05-22 09:55:32.5', '2025-05-22 09:55:30', '2025-05-22 09:55:40', '2025-05-22 09:55:37.5', '2025-05-22 09:55:47.5', '2025-05-22 09:56:45', '2025-05-22 09:56:55', '2025-05-22 09:56:52.5', '2025-05-22 09:57:02.5']

# crop 10s windows without overlap
clean_windows = {}
noisy_windows = {}
window_index = 1
while window_index < len(clean_timestamps):
    clean_windows[window_index] = crop_window(bp_clean, clean_timestamps[window_index - 1], clean_timestamps[window_index])
    noisy_windows[window_index] = crop_window(bp_noisy, noisy_timestamps[window_index - 1], noisy_timestamps[window_index])
    window_index += 1

# crop 10s windows with overlap
overlapping_clean_windows = {}
overlapping_noisy_windows = {}
window_index = 1
while window_index < len(overlapping_clean_timestamps):
    overlapping_clean_windows[window_index] = crop_window(bp_clean, overlapping_clean_timestamps[window_index - 1], overlapping_clean_timestamps[window_index])
    overlapping_noisy_windows[window_index] = crop_window(bp_noisy, overlapping_noisy_timestamps[window_index - 1], overlapping_noisy_timestamps[window_index])
    window_index += 2



In [ ]:
# calculate measurements for overlapping clean windows
overlap_clean_windows_df = pd.DataFrame({
    'Segment': [], 'Mean': [], 'Median': [], 'Variance': [], 'Std Dev': [], 'Skewness': [], 'RMS': [], 'Zero-Crossing Rate': [],
    'Energy': [], 'Power': []
})

for window in overlapping_clean_windows:
    overlap_clean_windows_df.loc[len(overlap_clean_windows_df)] = [
        int(window/2),
        overlapping_clean_windows[window]['ecg'].mean(),
        overlapping_clean_windows[window]['ecg'].median(),
        overlapping_clean_windows[window]['ecg'].var(),
        overlapping_clean_windows[window]['ecg'].std(),
        overlapping_clean_windows[window]['ecg'].skew(),
        np.sqrt(np.mean(overlapping_clean_windows[window]['ecg'] ** 2)),
        np.mean(np.diff(np.sign(overlapping_clean_windows[window]['ecg'])) != 0),
        np.sum(overlapping_clean_windows[window]['ecg'] ** 2),
        np.mean(overlapping_clean_windows[window]['ecg'] ** 2)
    ]

# print results for q1b, part 1/4
overlap_clean_windows_df

In [ ]:
# calculate measurements for overlapping noisy windows
overlap_noisy_windows_df = pd.DataFrame({
    'Segment': [], 'Mean': [], 'Median': [], 'Variance': [], 'Std Dev': [], 'Skewness': [], 'RMS': [], 'Zero-Crossing Rate': [],
    'Energy': [], 'Power': []
})

for window in overlapping_noisy_windows:
    overlap_noisy_windows_df.loc[len(overlap_noisy_windows_df)] = [
        window/2,
        overlapping_noisy_windows[window]['ecg'].mean(),
        overlapping_noisy_windows[window]['ecg'].median(),
        overlapping_noisy_windows[window]['ecg'].var(),
        overlapping_noisy_windows[window]['ecg'].std(),
        overlapping_noisy_windows[window]['ecg'].skew(),
        np.sqrt(np.mean(overlapping_noisy_windows[window]['ecg'] ** 2)),
        np.mean(np.diff(np.sign(overlapping_noisy_windows[window]['ecg'])) != 0),
        np.sum(overlapping_noisy_windows[window]['ecg'] ** 2),
        np.mean(overlapping_noisy_windows[window]['ecg'] ** 2)
    ]

# print results for q1b, part 2/4
overlap_noisy_windows_df

In [ ]:
# calculate measurements for no overlap clean windows
no_overlap_clean_windows_df = pd.DataFrame({
    'Segment': [], 'Mean': [], 'Median': [], 'Variance': [], 'Std Dev': [], 'Skewness': [], 'RMS': [], 'Zero-Crossing Rate': [],
    'Energy': [], 'Power': []
})

for window in clean_windows:
    no_overlap_clean_windows_df.loc[len(no_overlap_clean_windows_df)] = [
        window,
        clean_windows[window]['ecg'].mean(),
        clean_windows[window]['ecg'].median(),
        clean_windows[window]['ecg'].var(),
        clean_windows[window]['ecg'].std(),
        clean_windows[window]['ecg'].skew(),
        np.sqrt(np.mean(clean_windows[window]['ecg'] ** 2)),
        np.mean(np.diff(np.sign(clean_windows[window]['ecg'])) != 0),
        np.sum(clean_windows[window]['ecg'] ** 2),
        np.mean(clean_windows[window]['ecg'] ** 2)
    ]

# print results for q1b, part 3/4
no_overlap_clean_windows_df

In [ ]:
# calculate measurements for non-overlapping noisy windows
no_overlap_noisy_windows_df = pd.DataFrame({
    'Segment': [], 'Mean': [], 'Median': [], 'Variance': [], 'Std Dev': [], 'Skewness': [], 'RMS': [], 'Zero-Crossing Rate': [],
    'Energy': [], 'Power': []
})

for window in noisy_windows:
    no_overlap_noisy_windows_df.loc[len(no_overlap_noisy_windows_df)] = [
        window,
        noisy_windows[window]['ecg'].mean(),
        noisy_windows[window]['ecg'].median(),
        noisy_windows[window]['ecg'].var(),
        noisy_windows[window]['ecg'].std(),
        noisy_windows[window]['ecg'].skew(),
        np.sqrt(np.mean(noisy_windows[window]['ecg'] ** 2)),
        np.mean(np.diff(np.sign(noisy_windows[window]['ecg'])) != 0),
        np.sum(noisy_windows[window]['ecg'] ** 2),
        np.mean(noisy_windows[window]['ecg'] ** 2)
    ]

# print results for q1b, part 4/4
no_overlap_noisy_windows_df

In [ ]:
# Q2a

# crop 10s windows
dft_window_clean = crop_window(bp_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')
dft_window_noisy = crop_window(bp_noisy, '2025-05-22 09:55:00', '2025-05-22 09:55:10')

dft_clean = np.fft.fft(dft_window_clean['ecg'])
plt.plot(dft_clean)
plt.title('DFT of 10s Window of Clean ECG')
plt.ylabel('DFT Amplitude')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

dft_noisy = np.fft.fft(dft_window_noisy['ecg'])
plt.plot(dft_noisy)
plt.title('DFT of 10s Window of Noisy ECG')
plt.ylabel('DFT Amplitude')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


In [ ]:
#Q2b
plt.plot(scipy.signal.periodogram(dft_window_clean['ecg'])[1])
plt.title('Periodogram of 10s Window of Clean ECG from BioPac')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

clean_welch_bp = scipy.signal.welch(dft_window_clean['ecg'])[1]
plt.plot(clean_welch_bp)
plt.title('Welch of 10s Window of Clean ECG from BioPac')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


plt.plot(scipy.signal.periodogram(dft_window_noisy['ecg'])[1])
plt.title('Periodogram of 10s Window of Noisy ECG from BioPac')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

noisy_welch_bp = scipy.signal.welch(dft_window_noisy['ecg'])[1]
plt.plot(noisy_welch_bp)
plt.title('Welch of 10s Window of Noisy ECG from BioPac')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

print(f'Centroid (clean bp): {scipy.ndimage.center_of_mass(clean_welch_bp)[0]}')
print(f'Centroid (noisy bp): {scipy.ndimage.center_of_mass(noisy_welch_bp)[0]}')

print(f'Entropy (clean bp): {scipy.stats.entropy(clean_welch_bp)}')
print(f'Entropy (noisy bp): {scipy.stats.entropy(noisy_welch_bp)}')




In [ ]:
#Q2b part 2, same thing but for watch data for the last part of the exercise

# crop 10s windows
dft_window_clean = crop_window(watch_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')
dft_window_noisy = crop_window(watch_noisy, '2025-05-22 09:55:00', '2025-05-22 09:55:10')

plt.plot(scipy.signal.periodogram(dft_window_clean['ecg'])[1])
plt.title('Periodogram of 10s Window of Clean ECG from Smartwatch')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

clean_welch_bp = scipy.signal.welch(dft_window_clean['ecg'])[1]
plt.plot(clean_welch_bp)
plt.title('Welch of 10s Window of Clean ECG from Smartwatch')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


plt.plot(scipy.signal.periodogram(dft_window_noisy['ecg'])[1])
plt.title('Periodogram of 10s Window of Noisy ECG from Smartwatch')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

noisy_welch_bp = scipy.signal.welch(dft_window_noisy['ecg'])[1]
plt.plot(noisy_welch_bp)
plt.title('Welch of 10s Window of Noisy ECG from Smartwatch')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

print(f'Centroid (clean watch): {scipy.ndimage.center_of_mass(clean_welch_bp)[0]}')
print(f'Centroid (noisy watch): {scipy.ndimage.center_of_mass(noisy_welch_bp)[0]}')

print(f'Entropy (clean watch): {scipy.stats.entropy(clean_welch_bp)}')
print(f'Entropy (noisy watch): {scipy.stats.entropy(noisy_welch_bp)}')



In [ ]:
# Q3a

# crop 10s windows
dft_window_clean = crop_window(watch_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')
dft_window_noisy = crop_window(watch_motion, '2025-05-22 10:04:00', '2025-05-22 10:04:10')

dft_clean = np.fft.fft(dft_window_clean['PPG'])
plt.plot(dft_clean)
plt.title('DFT of 10s Window of Clean PPG from Smartwatch')
plt.ylabel('DFT Amplitude')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

dft_noisy = np.fft.fft(dft_window_noisy['PPG'])
plt.plot(dft_noisy)
plt.title('DFT of 10s Window of motion PPG from Smartwatch')
plt.ylabel('DFT Amplitude')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

In [ ]:
# Q3b

plt.plot(scipy.signal.periodogram(dft_window_clean['PPG'])[1])
plt.title('Periodogram of 10s Window of Clean ECG from BioPac')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

clean_welch_bp = scipy.signal.welch(dft_window_clean['PPG'])[1]
plt.plot(clean_welch_bp)
plt.title('Welch of 10s Window of Clean PPG from Smartwatch')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()


plt.plot(scipy.signal.periodogram(dft_window_noisy['PPG'])[1])
plt.title('Periodogram of 10s Window of Noisy PPG from Smartwatch')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

noisy_welch_bp = scipy.signal.welch(dft_window_noisy['PPG'])[1]
plt.plot(noisy_welch_bp)
plt.title('Welch of 10s Window of Noisy ECG from Smartwatch')
plt.ylabel('Power (V^2/Hz)')
plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

print(f'Centroid (clean watch): {scipy.ndimage.center_of_mass(clean_welch_bp)[0]}')
print(f'Centroid (noisy watch): {scipy.ndimage.center_of_mass(noisy_welch_bp)[0]}')

print(f'Entropy (clean watch): {scipy.stats.entropy(clean_welch_bp)}')
print(f'Entropy (noisy watch): {scipy.stats.entropy(noisy_welch_bp)}')

# Time-frequency domain


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def generate_stationary_signals(fs, duration, frequencies):
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    return [np.sin(2 * np.pi * freq * t) for freq in frequencies]


def generate_composite_signal(fs, duration, frequencies):
    signal_comp = np.sum(np.stack(generate_stationary_signals(fs, duration, frequencies)), axis=0)
    return signal_comp


def generate_non_stationary_signals(fs,  # Sampling frequency (Hz)
                                    frequencies,  # Frequencies to generate
                                    cycles):
    # Calculate durations and create time array
    segment_durations = [c / f for c, f in zip(cycles, frequencies)]
    total_duration = sum(segment_durations)
    t = np.linspace(0, total_duration, int(fs * total_duration), endpoint=False)
    signal_non_st = np.zeros(len(t))

    # Generate a Non-Stationary signal 
    current_sample = 0
    for freq, n_cycles in zip(frequencies, cycles):
        seg_samples = int(n_cycles / freq * fs)
        local_t = np.arange(seg_samples) / fs
        segment = np.sin(2 * np.pi * freq * local_t)
        signal_non_st[current_sample:current_sample + seg_samples] = segment
        current_sample += seg_samples

    # Create reversed signal
    reversed_signal_non_st = signal_non_st[::-1]

    return signal_non_st, reversed_signal_non_st


stationary_signals = generate_stationary_signals(fs=500, duration=1, frequencies=[10, 25, 50, 100])
comp_signal = generate_composite_signal(fs=500, duration=1, frequencies=[10, 25, 50, 100])
signal_non_st, reversed_signal_non_st = generate_non_stationary_signals(fs=500,  # Sampling frequency (Hz)
                                                                        frequencies=[10, 25, 50, 100],
                                                                        # Frequencies to generate
                                                                        cycles=[3, 6, 12, 25])
pass

In [ ]:
# Q4
print('We only print the comp_signal and signal_non_st, since the others are redundant and stationary signals are a lot of simple sinus curves with a simple difference in freq.')
signals = {
    'Composed Signal': comp_signal,
    'Signal non Stationary': signal_non_st,
    'Signal non Stationary Reversed': reversed_signal_non_st
}

def compute_and_plot_spectrum(signal, title):
    t = np.linspace(0, 1, len(signal))
    fs = 500
    n = len(signal)
    freqs = np.fft.fftfreq(n, d=1/fs)
    spectrum = np.fft.fft(signal)
    power = np.abs(spectrum)**2 / n
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(t, signal)
    plt.title(f"{title} - Signal")
    plt.subplot(1, 2, 2)
    plt.plot(freqs[:n//2], power[:n//2])
    plt.title(f"{title} - Power Spectrum")
    plt.tight_layout()
    plt.show()

for name, signal in signals.items():
    compute_and_plot_spectrum(signal, name)

### Fast Fourier Transform (FFT) of the signals

In [ ]:
# Q5

# Frequencies to generate
signals = {
'signal_non_st': signal_non_st,
'reversed_signal_non_st': reversed_signal_non_st
}
for name, signal in signals.items():
    plt.figure(figsize=(10, 4))
    plt.plot(signal)
    plt.title(f'{name} - Time Domain Signal')
    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.grid()
    plt.tight_layout()
    plt.show()

    window_size = 128
    hop_size = 128
    nfft = 1024  # FFT size
    fs=500

    # Hann window
    #window = np.hanning(window_size)

    # Calculate number of frames
    num_frames = (len(signal) - window_size) // hop_size + 1
    spectrogram = np.zeros((nfft // 2, num_frames))

    for i in range(num_frames):
        start = i * hop_size
        segment = signal[start:start + window_size]
        spectrum = np.fft.fft(segment, n=nfft)[:nfft // 2]
        spectrogram[:, i] = np.abs(spectrum)

    # Time and frequency axes
    time = np.arange(num_frames) * hop_size / fs
    freq = np.fft.fftfreq(nfft, d=1/fs)[:nfft // 2]

    plt.figure(figsize=(10, 6))
    plt.imshow(20 * np.log10(spectrogram + 1e-6), aspect='auto', origin='lower',
    extent=[time[0], time[-1], freq[0], freq[-1]], cmap='jet')
    plt.colorbar(label='Magnitude (dB)')
    plt.title(name + ' - STFT Magnitude')
    plt.xlabel('Time [s]')
    plt.ylabel('Frequency [Hz]')
    plt.tight_layout()
    plt.show()



# Time-frequency-domain analysis

In [ ]:
# Q6 a Ten Second ECG with and without noise
dft_window_clean = crop_window(bp_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')
dft_window_noisy = crop_window(bp_noisy, '2025-05-22 09:55:00', '2025-05-22 09:55:10')

def plot_stft(signal, fs, window_size=128, hop_size=128, nfft=1024, name='Signal'):

    num_frames = (len(signal) - window_size) // hop_size + 1
    spectrogram = np.zeros((nfft // 2, num_frames))

    for i in range(num_frames):
        start = i * hop_size
        segment = signal[start:start + window_size]
        spectrum = np.fft.fft(segment, n=nfft)[:nfft // 2]
        spectrogram[:, i] = np.abs(spectrum)

    time = np.arange(num_frames) * hop_size / fs
    freq = np.fft.fftfreq(nfft, d=1/fs)[:nfft // 2]

    # Plot Signal
    plt.figure(figsize=(10,5))
    plt.plot(signal)
    plt.title('Original Signal - ' + name)
    plt.xlabel('Time [ms]')
    plt.ylabel('Value')
    plt.tight_layout()
    plt.show()


    plt.figure(figsize=(10, 6))
    plt.imshow(20 * np.log10(spectrogram + 1e-6), aspect='auto', origin='lower',
               extent=[time[0], time[-1], freq[0], freq[-1]], cmap='jet')
    plt.colorbar(label='Magnitude (dB)')
    plt.title('STFT Magnitude - ' + name)
    plt.xlabel('Time [s]')
    plt.ylabel('Frequency [Hz]')
    plt.tight_layout()
    plt.show()


plot_stft(dft_window_clean['ecg'].values, fs=500, window_size=128, hop_size=128, nfft=1024, name='Clean ECG')
plot_stft(dft_window_noisy['ecg'].values, fs=500, window_size=128, hop_size=128, nfft=1024, name='Noisy ECG')


In [ ]:
# Q6 b Ten Second PPG of both devices and with motion noise
bp_window_motion = crop_window(watch_motion, '2025-05-22 10:04:20', '2025-05-22 10:04:30')
watch_window_motion = crop_window(bp_motion, '2025-05-22 10:04:20', '2025-05-22 10:04:30')

plot_stft(bp_window_motion['PPG'].values, fs=500, window_size=128, hop_size=128, nfft=1024, name='BioPac PPG with Motion')
plot_stft(watch_window_motion['ppg'].values, fs=500, window_size=128, hop_size=128, nfft=1024, name='Watch PPG with Motion')


In [ ]:
# Q7 a CWT
dft_window_clean = crop_window(bp_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')
dft_window_noisy = crop_window(bp_noisy, '2025-05-22 09:55:00', '2025-05-22 09:55:10')

wavelet = 'cmor1.5-1.0'  # Complex Morlet wavelet: good for time-frequency analysis
scales = np.arange(1, 128)  # Adjust scale range based on frequency content

data = {
    'Clean': dft_window_clean['ecg'].values,
    'Noisy': dft_window_noisy['ecg'].values
}

def plot_cwt(signal, name, wavelet='cmor1.5-1.0', scales=np.arange(1,128), appendix='ECG'):
    coefficients, frequencies = pywt.cwt(signal, scales, wavelet, sampling_period=1/500)

    # Plot scalogram
    plt.figure(figsize=(10,5))
    plt.plot(np.linspace(0, 10, 10*1000), signal)
    plt.title('Original Signal - ' + name)
    plt.xlabel('Time [s]')
    plt.ylabel('Value')
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(10, 6))
    plt.imshow(np.abs(coefficients), extent=[0, 10, frequencies[-1], frequencies[0]],
               cmap='viridis', aspect='auto')
    plt.colorbar(label='Magnitude')
    plt.xlabel('Time [s]')
    plt.ylabel('Frequency [Hz]')
    plt.title(f'CWT (Scalogram) - {name} {appendix}')
    plt.tight_layout()
    plt.show()


for name, signal in data.items():
    # Perform CWT
    plot_cwt(signal[:-1], name + ' wavelet gaus1', wavelet='gaus1')
    plot_cwt(signal[:-1], name + ' wavelet cmor1.5-1.0', wavelet='cmor1.5-1.0')



In [ ]:
#Q7 b PPG
watch_clean_window = crop_window(watch_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')
bp_clean_window = crop_window(bp_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')

# STD norm Watch
signal = watch_clean_window['PPG'][:10000]
#signal = (signal - np.mean(signal))/np.std(signal)
#signal = (signal - np.min(signal))/ np.max(signal)

plot_cwt(bp_clean_window['ppg'][:-1], name='BioPac in Motion, wavelet cmor', appendix='PPG', wavelet='cmor1.5-1.0')
plot_cwt(signal, name='Watch in Motion, wavelet cmor', appendix='PPG', wavelet='cmor1.5-1.0')

In [ ]:
# Q8 a DWT
dft_window_clean = crop_window(bp_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')
dft_window_noisy = crop_window(bp_noisy, '2025-05-22 09:55:00', '2025-05-22 09:55:10')

def plot_dwt(signal, name, wavelet='db4'):
    coeffs = pywt.dwt(signal, wavelet)

    cA, cD = coeffs  # Approximation and detail coefficients

    # Plot
    plt.figure(figsize=(10, 6))
    plt.subplot(3, 1, 1)
    t = np.linspace(0, 10, 10*1000)
    plt.plot(t, signal)
    plt.title("Original Signal - " + name)
    plt.xlabel('Time [s]')
    plt.ylabel('value')

    plt.subplot(3, 1, 2)
    plt.plot(cA)
    plt.title("Approximation Coefficients - " + name)
    plt.xlabel('Sample')
    plt.ylabel('value')


    plt.subplot(3, 1, 3)
    plt.plot(cD)
    plt.title("Detail Coefficients - " + name)
    plt.xlabel('Sample')
    plt.ylabel('value')

    plt.tight_layout()
    plt.show()

plot_dwt(dft_window_clean['ecg'][:10000], name='Clean', wavelet='db3')
plot_dwt(dft_window_noisy['ecg'][:10000], name='Noisy', wavelet='db3')

In [ ]:
# Q8 b DWT PPG
watch_clean_window = crop_window(watch_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')
bp_clean_window = crop_window(bp_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:10')


plot_dwt(watch_clean_window['PPG'][:10000], name='PPG Watch clean', wavelet='db4')
plot_dwt(bp_clean_window['ppg'][:10000], name='PPG BP clean', wavelet='db3')


In [ ]:
# Q9 a
print('Use 10sec Window for better visibility')
bp_clean_window = crop_window(bp_clean, '2025-05-22 09:45:00', '2025-05-22 09:45:20')
ecg = bp_clean_window['ecg'][:-1]
t = np.linspace(0, 10, 20*1000)

wavelet = 'db4'
max_level = pywt.dwt_max_level(len(ecg), pywt.Wavelet(wavelet).dec_len)

# b. Perform DWT decomposition and plot
coeffs = pywt.wavedec(ecg, wavelet, level=max_level)
plt.figure(figsize=(20, 15))
plt.subplot(len(coeffs)+1, 1, 1)
plt.plot(t, ecg)
plt.xlabel('Time [s]')
plt.ylabel('Value')
plt.title('Original ECG Signal (Clean BioPac Signal)')
for i, c in enumerate(coeffs):
    plt.subplot(len(coeffs)+1, 1, i+2)
    plt.plot(c)
    plt.title(f'Level {i} Coefficients')
plt.tight_layout()
plt.show()

# c. Reconstruct baseline (keep approximation, zero detail)
approx_only = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
baseline = pywt.waverec(approx_only, wavelet)
plt.figure(figsize=(20, 5))
plt.plot(t, baseline[:len(t)])
plt.title('Reconstructed Baseline')
plt.tight_layout()
plt.xlabel('Time [s]')
plt.ylabel('Value')
plt.show()

# d. Reconstruct signal without baseline (zero approximation, keep details)
details_only = [np.zeros_like(coeffs[0])] + coeffs[1:]
ecg_no_baseline = pywt.waverec(details_only, wavelet)
plt.figure(figsize=(20, 5))
plt.plot(t, ecg, label='Original ECG (clean)')
plt.plot(t, ecg_no_baseline[:len(t)], label='ECG without Baseline')
plt.legend()
plt.xlabel('Time [s]')
plt.ylabel('Value')
plt.title('Baseline Removed ECG')
plt.tight_layout()
plt.show()

# e. Experiment with wavelets and levels
wavelets = ['db4', 'sym4', 'bior3.9']
levels = [6, 8, 10]

for wave in wavelets:
    for lvl in levels:
        max_lvl = pywt.dwt_max_level(len(ecg), pywt.Wavelet(wave).dec_len)
        lvl = min(lvl, max_lvl)
        coeffs = pywt.wavedec(ecg, wave, level=lvl)
        details_only = [np.zeros_like(coeffs[0])] + coeffs[1:]
        ecg_no_baseline = pywt.waverec(details_only, wave)
        plt.figure(figsize=(20, 5))
        plt.plot(t, ecg, label='Original ECG')
        plt.plot(t, ecg_no_baseline[:len(t)], label=f'{wave}, level={lvl}')
        plt.title(f'Baseline Removal using {wave}, level {lvl}')
        plt.xlabel('Time [s]')
        plt.ylabel('Value')
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
# Q10
bp_clean_window = crop_window(bp_noisy, '2025-05-22 09:55:10', '2025-05-22 09:55:30')
ecg = bp_clean_window['ecg'][:-1]
t = np.linspace(0, 10, 20*1000)

wavelet = 'db4'
max_level = pywt.dwt_max_level(len(ecg), pywt.Wavelet(wavelet).dec_len)

# b. Perform DWT decomposition and plot
coeffs = pywt.wavedec(ecg, wavelet, level=max_level)
plt.figure(figsize=(20, 15))
plt.subplot(len(coeffs)+1, 1, 1)
plt.plot(t, ecg)
plt.title('Original Noisy ECG Signal')
plt.xlabel('Time [s]')
plt.ylabel('Value')
for i, c in enumerate(coeffs):
    plt.subplot(len(coeffs)+1, 1, i+2)
    plt.plot(c)
    plt.title(f'Level {i} Coefficients')
    plt.xlabel('Time [s]')
    plt.ylabel('Value')
plt.tight_layout()
plt.show()

# c. Zeroing D1, D2, D3 individually and reconstructing
for i in range(1, 4):
    filtered_coeffs = coeffs.copy()
    filtered_coeffs[i] = np.zeros_like(filtered_coeffs[i])
    filtered_signal = pywt.waverec(filtered_coeffs, wavelet)
    plt.figure(figsize=(20, 5))
    plt.plot(t, ecg, label='Original ECG', alpha=0.5)
    plt.plot(t, filtered_signal[:len(t)], label=f'D{i} zeroed')
    plt.title(f'Denoising by Zeroing D{i}')
    plt.xlabel('Time [s]')
    plt.ylabel('Value')
    plt.legend()
    plt.show()

# d. Experiment with different wavelets and levels
wavelets = ['db4', 'sym4', 'bior3.9']
levels = [6, 8, 10]

for wave in wavelets:
    for lvl in levels:
        max_lvl = pywt.dwt_max_level(len(ecg), pywt.Wavelet(wave).dec_len)
        lvl = min(lvl, max_lvl)
        coeffs = pywt.wavedec(ecg, wave, level=lvl)
        # Zero all detail coefficients
        filtered_coeffs = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
        filtered_signal = pywt.waverec(filtered_coeffs, wave)
        plt.figure(figsize=(20, 5))
        plt.plot(t, ecg, label='Original ECG', alpha=0.5)
        plt.plot(t, filtered_signal[:len(t)], label=f'{wave}, level={lvl}')
        plt.title(f'Denoised ECG using {wave} at level {lvl}')
        plt.xlabel('Time [s]')
        plt.ylabel('Value')
        plt.legend()
        plt.show()